# Week6-Exercise - Better Prediction without Fine-tuning a Frontier Model

I extended this week's work with sampling hyperparameters. As seen earlier, The Frontier models predict better without Fine-tuning as our training data made them perform poorly.


Ed noted that prompt can be better used in our case than fine-tuning. I decided to take this further 
by utilizing sampling hyperparameters (temperature and top_k parameters) to make Frontier models deterministic and focused.
This shows that models predicts much more closer to the current prices of the products as a result of getting more deterministics.


In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item

In [ ]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

In [6]:
#user message
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [ ]:
print(test[0].summary)

In [ ]:
messages_for(test[0])

In [8]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item), temperature=0.0, top_p=0.1)
    return response.choices[0].message.content

In [ ]:
gpt_4__1_nano(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(gpt_4__1_nano, test, size=20, workers=1)

In [26]:
def gemini_2__5_flash(item):
    response = completion(model="gemini/gemini-2.5-flash", messages=messages_for(item), temperature=0.0, top_p=0.1)
    return response.choices[0].message.content

In [ ]:
gemini_2__5_flash(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(gemini_2__5_flash, test, size=20, workers=1)